In [0]:
-- ============================================================
-- PROJECT   : Retail Data Warehouse — Databricks Lakeflow
-- FILE      : silver_validation_queries.sql
-- AUTHOR    : Ahsan Ali Abbasi
-- LAYER     : Bronze → Silver Validation & Data Quality Checks
-- PURPOSE   : End-to-end validation of bronze ingestion and
--             silver transformation layer. Covers row count
--             reconciliation, null checks, domain validation,
--             referential integrity, deduplication, and
--             advanced SQL patterns (CASE, CTEs, window
--             functions, date intervals).
-- ============================================================


-- ============================================================
-- SECTION 1: BRONZE LAYER — ROW COUNT VALIDATION
-- Verifies all 9 source tables were fully ingested from raw
-- CSV files into the bronze streaming tables. Expected counts
-- are defined by the source dataset specification.
-- ============================================================

SELECT 'bronze_customers'               AS table_name, COUNT(*) AS row_count FROM workspace.default.bronze_customers
UNION ALL
SELECT 'bronze_orders',                  COUNT(*) FROM workspace.default.bronze_orders
UNION ALL
SELECT 'bronze_employees',               COUNT(*) FROM workspace.default.bronze_employees
UNION ALL
SELECT 'bronze_products',                COUNT(*) FROM workspace.default.bronze_products
UNION ALL
SELECT 'bronze_stores',                  COUNT(*) FROM workspace.default.bronze_stores
UNION ALL
SELECT 'bronze_returns',                 COUNT(*) FROM workspace.default.bronze_returns
UNION ALL
SELECT 'bronze_inventory_snapshots',     COUNT(*) FROM workspace.default.bronze_inventory_snapshots
UNION ALL
SELECT 'bronze_customer_segment_bridge', COUNT(*) FROM workspace.default.bronze_customer_segment_bridge
UNION ALL
SELECT 'bronze_order_items',             COUNT(*) FROM workspace.default.bronze_order_items
ORDER BY table_name;

-- Expected Results:
-- bronze_customer_segment_bridge : 658
-- bronze_customers               : 386
-- bronze_employees               : 40
-- bronze_inventory_snapshots     : 4500
-- bronze_order_items             : 5996
-- bronze_orders                  : 2000
-- bronze_products                : 99
-- bronze_returns                 : 225
-- bronze_stores                  : 15


-- ============================================================
-- SECTION 2: BRONZE LAYER — SOURCE SYSTEM AUDIT
-- Confirms multi-source customer data (CRM, Ecommerce, POS)
-- was ingested correctly and audit columns (source_file,
-- ingested_at) are populated.
-- ============================================================

-- Distribution of customer records across source systems
-- Validates that all 3 operational systems contributed data
SELECT
  source_system,
  COUNT(*) AS record_count
FROM workspace.default.bronze_customers
GROUP BY source_system
ORDER BY record_count DESC;

-- Confirm order statuses are within expected domain values
-- Should only contain: Completed, Cancelled, Pending
SELECT
  order_status,
  COUNT(order_id) AS order_count
FROM workspace.default.bronze_orders
GROUP BY order_status
ORDER BY order_count DESC;

-- Verify 12 monthly inventory snapshots loaded correctly
-- Each snapshot_date should have one row per store/product combination
SELECT
  snapshot_date,
  COUNT(*) AS store_product_combinations
FROM workspace.default.bronze_inventory_snapshots
GROUP BY snapshot_date
ORDER BY snapshot_date;

-- Verify audit columns are correctly populated
-- source_file should show the exact Volume path the file was read from
-- ingested_at should show a recent timestamp
SELECT DISTINCT
  source_file,
  ingested_at
FROM workspace.default.bronze_customers;


-- ============================================================
-- SECTION 3: SILVER LAYER — BRONZE VS SILVER RECONCILIATION
-- Compares row counts between bronze and silver for every
-- table. Silver should have equal or fewer rows than bronze
-- because EXPECT constraints drop rows that fail validation.
-- The DeltaCount column shows exactly how many rows were
-- removed by data quality constraints in the silver layer.
-- ============================================================

SELECT
  'Employees'          AS TableName,
  COUNT(*)             AS Bronze FROM workspace.default.bronze_employees,
  COUNT(*)             AS Silver FROM workspace.default.silver_employees,
  (Bronze - Silver)    AS DeltaCount
UNION ALL
SELECT
  'Products',
  (SELECT COUNT(*) FROM workspace.default.bronze_products),
  (SELECT COUNT(*) FROM workspace.default.silver_products),
  (SELECT COUNT(*) FROM workspace.default.bronze_products) -
  (SELECT COUNT(*) FROM workspace.default.silver_products)
UNION ALL
SELECT
  'Orders',
  (SELECT COUNT(*) FROM workspace.default.bronze_orders),
  (SELECT COUNT(*) FROM workspace.default.silver_orders),
  (SELECT COUNT(*) FROM workspace.default.bronze_orders) -
  (SELECT COUNT(*) FROM workspace.default.silver_orders)
UNION ALL
SELECT
  'Order Items',
  (SELECT COUNT(*) FROM workspace.default.bronze_order_items),
  (SELECT COUNT(*) FROM workspace.default.silver_order_items),
  (SELECT COUNT(*) FROM workspace.default.bronze_order_items) -
  (SELECT COUNT(*) FROM workspace.default.silver_order_items)
UNION ALL
SELECT
  'Stores',
  (SELECT COUNT(*) FROM workspace.default.bronze_stores),
  (SELECT COUNT(*) FROM workspace.default.silver_stores),
  (SELECT COUNT(*) FROM workspace.default.bronze_stores) -
  (SELECT COUNT(*) FROM workspace.default.silver_stores)
UNION ALL
SELECT
  'Returns',
  (SELECT COUNT(*) FROM workspace.default.bronze_returns),
  (SELECT COUNT(*) FROM workspace.default.silver_returns),
  (SELECT COUNT(*) FROM workspace.default.bronze_returns) -
  (SELECT COUNT(*) FROM workspace.default.silver_returns)
UNION ALL
SELECT
  'Inventory Snapshots',
  (SELECT COUNT(*) FROM workspace.default.bronze_inventory_snapshots),
  (SELECT COUNT(*) FROM workspace.default.silver_inventory_snapshots),
  (SELECT COUNT(*) FROM workspace.default.bronze_inventory_snapshots) -
  (SELECT COUNT(*) FROM workspace.default.silver_inventory_snapshots)
UNION ALL
SELECT
  'Customer Segments',
  (SELECT COUNT(*) FROM workspace.default.bronze_customer_segment_bridge),
  (SELECT COUNT(*) FROM workspace.default.silver_customer_segment_bridge),
  (SELECT COUNT(*) FROM workspace.default.bronze_customer_segment_bridge) -
  (SELECT COUNT(*) FROM workspace.default.silver_customer_segment_bridge);


-- ============================================================
-- SECTION 4: SILVER LAYER — NULL CHECKS
-- Checks every column in silver_orders for NULL values.
-- ship_date and delivery_date are expected to have NULLs
-- (cancelled orders have no ship date, pending orders have
-- no delivery date). All other columns should return 0.
-- ============================================================

SELECT
  SUM(CASE WHEN order_id        IS NULL THEN 1 ELSE 0 END) AS null_order_id,
  SUM(CASE WHEN customer_src_id IS NULL THEN 1 ELSE 0 END) AS null_customer_src_id,
  SUM(CASE WHEN store_id        IS NULL THEN 1 ELSE 0 END) AS null_store_id,
  SUM(CASE WHEN employee_id     IS NULL THEN 1 ELSE 0 END) AS null_employee_id,
  SUM(CASE WHEN order_date      IS NULL THEN 1 ELSE 0 END) AS null_order_date,
  SUM(CASE WHEN ship_date       IS NULL THEN 1 ELSE 0 END) AS null_ship_date,        -- NULLs expected (cancelled)
  SUM(CASE WHEN delivery_date   IS NULL THEN 1 ELSE 0 END) AS null_delivery_date,    -- NULLs expected (pending)
  SUM(CASE WHEN order_status    IS NULL THEN 1 ELSE 0 END) AS null_order_status,
  SUM(CASE WHEN payment_method  IS NULL THEN 1 ELSE 0 END) AS null_payment_method
FROM workspace.default.silver_orders;


-- ============================================================
-- SECTION 5: SILVER LAYER — NET AMOUNT CALCULATION AUDIT
-- Validates that the derived column net_amount in
-- silver_order_items was computed correctly using the formula:
--   net_amount = quantity × unit_price_at_sale × (1 − discount_pct)
-- Any row showing 'MISMATCH' indicates a calculation error.
-- Expected: all rows return 'MATCH'.
-- ============================================================

SELECT
  order_item_id,
  order_id,
  product_src_id,
  quantity,
  unit_price_at_sale,
  discount_pct,
  net_amount                                                        AS stored_net_amount,
  ROUND((quantity * unit_price_at_sale) * (1 - discount_pct), 2)   AS expected_net_amount,
  CASE
    WHEN net_amount = ROUND((quantity * unit_price_at_sale) * (1 - discount_pct), 2)
    THEN 'MATCH'
    ELSE 'MISMATCH'
  END                                                               AS validation_status
FROM workspace.default.silver_order_items
ORDER BY validation_status DESC;  -- MISMATCHes appear first if any exist


-- ============================================================
-- SECTION 6: SILVER LAYER — DOMAIN VALIDATION
-- Checks that values fall within acceptable business domains.
-- Any count > 0 indicates data that slipped past EXPECT
-- constraints and requires investigation.
-- ============================================================

-- Confirm only valid order statuses exist after silver constraints
SELECT
  order_status,
  COUNT(*) AS cnt
FROM workspace.default.silver_orders
GROUP BY order_status;
-- Expected values only: Completed, Cancelled, Pending

-- Validate product pricing rules:
-- bad_prices: unit_price must be > 0
-- bad_costs: unit_cost must be > 0
-- negative_margin: price must always exceed cost (margin constraint)
SELECT
  SUM(CASE WHEN unit_price <= 0             THEN 1 ELSE 0 END) AS bad_prices,
  SUM(CASE WHEN unit_cost  <= 0             THEN 1 ELSE 0 END) AS bad_costs,
  SUM(CASE WHEN unit_price <= unit_cost     THEN 1 ELSE 0 END) AS negative_margin
FROM workspace.default.silver_products;
-- All three should return 0 — EXPECT constraints enforce this


-- ============================================================
-- SECTION 7: SILVER LAYER — REFERENTIAL INTEGRITY CHECKS
-- Identifies orphan records: rows that reference a parent
-- record that doesn't exist in the related table.
-- These are NOT dropped in silver — they are surfaced here
-- for awareness and handled in gold using the -1 Unknown
-- member strategy to preserve revenue reconciliation.
-- ============================================================

-- Orders referencing store_ids that don't exist in silver_stores
-- These came from the 6 deliberately injected RI breaks in raw data
SELECT
  o.order_id,
  o.store_id,
  o.order_date,
  o.order_status,
  'No matching store' AS issue
FROM workspace.default.silver_orders o
LEFT JOIN workspace.default.silver_stores s ON o.store_id = s.store_id
WHERE s.store_id IS NULL;

-- Returns referencing order_item_ids that don't exist in silver_order_items
-- These 5 rows were injected as deliberate RI breaks in raw_returns.csv
SELECT
  r.return_id,
  r.order_item_id,
  r.return_reason,
  r.refund_amount
FROM workspace.default.silver_returns r
LEFT JOIN workspace.default.silver_order_items oi ON r.order_item_id = oi.order_item_id
WHERE oi.order_item_id IS NULL;


-- ============================================================
-- SECTION 8: SILVER LAYER — DUPLICATE / PRIMARY KEY CHECKS
-- Primary keys must be unique. Any result here means a
-- duplicate ingestion occurred and the pipeline has a bug.
-- Expected: both queries return 0 rows.
-- ============================================================

-- order_id must be unique in silver_orders
SELECT
  order_id,
  COUNT(*) AS duplicate_count
FROM workspace.default.silver_orders
GROUP BY order_id
HAVING COUNT(*) > 1;

-- order_item_id must be unique in silver_order_items
SELECT
  order_item_id,
  COUNT(*) AS duplicate_count
FROM workspace.default.silver_order_items
GROUP BY order_item_id
HAVING COUNT(*) > 1;


-- ============================================================
-- SECTION 9: ADVANCED SQL — CASE STATEMENT
-- Classifies each order into a revenue size band using a
-- CASE expression. Groups by order_status AND band to show
-- how order value distribution differs by fulfillment status.
-- Demonstrates: CASE WHEN, JOIN, GROUP BY on derived column,
-- multiple aggregations in a single query.
-- ============================================================

SELECT
  so.order_id,
  so.order_status,
  SUM(soi.quantity)                   AS total_items,
  ROUND(SUM(soi.net_amount), 2)       AS total_revenue,
  ROUND(AVG(soi.net_amount), 2)       AS avg_item_value,
  ROUND(MIN(soi.net_amount), 2)       AS min_item_value,
  ROUND(MAX(soi.net_amount), 2)       AS max_item_value,
  CASE
    WHEN SUM(soi.net_amount) < 50   THEN 'Small    (< $50)'
    WHEN SUM(soi.net_amount) < 200  THEN 'Medium   ($50–$200)'
    WHEN SUM(soi.net_amount) <= 500 THEN 'Large    ($200–$500)'
    ELSE                                 'Enterprise (> $500)'
  END                                 AS order_classification
FROM workspace.default.silver_orders so
LEFT JOIN workspace.default.silver_order_items soi ON so.order_id = soi.order_id
GROUP BY so.order_id, so.order_status
ORDER BY total_revenue DESC;


-- ============================================================
-- SECTION 10: ADVANCED SQL — WINDOW FUNCTIONS
-- Ranks employees within each store by hire date (earliest
-- hire = rank 1). Also calculates total headcount per store
-- and each employee's tenure — all in a single scan without
-- collapsing rows.
-- Demonstrates: RANK() OVER (PARTITION BY), COUNT() OVER
-- (PARTITION BY), DATEDIFF, current_date().
-- ============================================================

SELECT
  employee_id,
  employee_name,
  role,
  store_id,
  hire_date,
  -- Rank 1 = earliest hire in the store (longest serving)
  RANK()   OVER (PARTITION BY store_id ORDER BY hire_date ASC)  AS hire_order_in_store,
  -- Total headcount per store (same value repeated on every row for that store)
  COUNT(*) OVER (PARTITION BY store_id)                         AS total_employees_in_store,
  -- Days between hire date and today
  DATEDIFF(current_date(), hire_date)                           AS days_employed,
  -- Rounded to 1 decimal for readability
  ROUND(DATEDIFF(current_date(), hire_date) / 365.0, 1)        AS years_employed
FROM workspace.default.silver_employees
ORDER BY store_id, hire_order_in_store;


-- ============================================================
-- SECTION 11: ADVANCED SQL — CHAINED CTEs
-- Finds the top 3 highest-margin products per category,
-- limited to only those products that beat their own
-- category's average margin. Uses 3 sequential CTEs where
-- each one builds on the previous result.
-- Demonstrates: chained CTEs, aggregation inside CTE,
-- JOIN between CTEs, window function inside CTE result,
-- filtering on window function output (rank <= 3).
-- ============================================================

WITH product_margins AS (
  -- CTE 1: Extract product margin details from silver
  SELECT
    product_src_id,
    product_name,
    category,
    brand,
    unit_price,
    unit_cost,
    gross_margin,
    margin_pct
  FROM workspace.default.silver_products
),
category_averages AS (
  -- CTE 2: Compute average, min, max margin per category
  SELECT
    category,
    ROUND(AVG(margin_pct), 4) AS avg_margin_pct,
    ROUND(MIN(margin_pct), 4) AS min_margin_pct,
    ROUND(MAX(margin_pct), 4) AS max_margin_pct,
    COUNT(*)                  AS total_products_in_category
  FROM product_margins
  GROUP BY category
),
above_average_ranked AS (
  -- CTE 3: Join products to their category stats, keep only
  -- products beating the category average, then rank them
  SELECT
    pm.product_name,
    pm.category,
    pm.brand,
    ROUND(pm.margin_pct, 4)                          AS product_margin_pct,
    ca.avg_margin_pct                                AS category_avg_margin_pct,
    ROUND(pm.margin_pct - ca.avg_margin_pct, 4)     AS margin_above_category_avg,
    ca.total_products_in_category,
    RANK() OVER (
      PARTITION BY pm.category
      ORDER BY pm.margin_pct DESC
    )                                                AS rank_in_category
  FROM product_margins pm
  JOIN category_averages ca ON pm.category = ca.category
  WHERE pm.margin_pct > ca.avg_margin_pct  -- only above-average products
)
-- Final: return top 3 per category only
SELECT *
FROM above_average_ranked
WHERE rank_in_category <= 3
ORDER BY category, rank_in_category;


-- ============================================================
-- SECTION 12: ADVANCED SQL — DATE INTERVALS + RUNNING TOTALS
-- Builds a complete monthly revenue trend report from daily
-- order data. Groups orders by month, calculates core KPIs,
-- then adds a cumulative running total and month-over-month
-- growth rate using window functions.
-- Demonstrates: DATE_TRUNC for month grouping, DATE_FORMAT
-- for readable labels, SUM OVER with ROWS BETWEEN for running
-- total, LAG for previous month comparison, NULLIF to prevent
-- division-by-zero in growth calculation.
-- ============================================================

WITH monthly_stats AS (
  -- CTE: Aggregate all order metrics at monthly grain
  SELECT
    DATE_TRUNC('month', o.order_date)    AS order_month,       -- for sorting
    DATE_FORMAT(o.order_date, 'yyyy-MM') AS month_label,       -- human-readable e.g. 2024-03
    COUNT(DISTINCT o.order_id)           AS total_orders,
    COUNT(DISTINCT o.customer_src_id)    AS unique_customers,
    ROUND(SUM(oi.net_amount), 2)         AS monthly_revenue,
    ROUND(AVG(oi.net_amount), 2)         AS avg_order_value,
    SUM(CASE WHEN o.order_status = 'Cancelled'
             THEN 1 ELSE 0 END)          AS cancelled_orders
  FROM workspace.default.silver_orders o
  JOIN workspace.default.silver_order_items oi ON o.order_id = oi.order_id
  GROUP BY
    DATE_TRUNC('month', o.order_date),
    DATE_FORMAT(o.order_date, 'yyyy-MM')
)
SELECT
  month_label,
  total_orders,
  unique_customers,
  monthly_revenue,
  avg_order_value,
  cancelled_orders,
  -- Cumulative revenue: sum of all months from start up to current row
  ROUND(SUM(monthly_revenue) OVER (
    ORDER BY order_month
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
  ), 2)                                  AS cumulative_revenue,
  -- Previous month's revenue for comparison
  LAG(monthly_revenue) OVER (
    ORDER BY order_month
  )                                      AS prev_month_revenue,
  -- Month-over-month growth % — NULLIF prevents division by zero
  -- when previous month revenue is 0
  ROUND(
    (monthly_revenue - LAG(monthly_revenue) OVER (ORDER BY order_month))
    / NULLIF(LAG(monthly_revenue) OVER (ORDER BY order_month), 0) * 100
  , 2)                                   AS mom_growth_pct
FROM monthly_stats
ORDER BY order_month;